# 04 · OpenVLA-OFT FFT 재로딩
소스는 Git에서, 가중치·데이터는 Hugging Face 고정 revision에서 받습니다. 실행 시 SOURCE_COMMIT을 기록하고 재개할 때 SOURCE_REF에 지정하십시오.
이 로컬 노트북 변경은 기존 Drive 노트북과 실행 중인 Colab에 자동 반영되지 않습니다.


In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, tempfile
SOURCE_REPOSITORY = 'https://github.com/eastha10/VLADE.git'
SOURCE_REF = globals().get('SOURCE_REF', 'main')  # 재개 시 이전 SOURCE_COMMIT을 지정하십시오.
ROOT = Path('/content/vlade_work')
ROOT.mkdir(parents=True, exist_ok=True)
def stage_source(repository: str, scratch_root: Path, source_ref: str = "main") -> Path:
    """Resolve a ref once and return a clean checkout of that immutable commit.

    Existing checkouts are never updated: another process may be using them.
    Git checks object integrity, so no per-file SHA256 inventory is needed.
    """
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]*", source_ref) or ".." in source_ref:
        raise ValueError("Invalid Git source reference")
    scratch_root = scratch_root.resolve()
    scratch_root.mkdir(parents=True, exist_ok=True)

    def git(directory, *args):
        return subprocess.run(["git", "-C", str(directory), *args], check=True,
                              capture_output=True, text=True).stdout.strip()

    with tempfile.TemporaryDirectory(prefix="vlade-fetch-", dir=scratch_root) as temporary:
        staging = Path(temporary) / "source"
        git(scratch_root, "clone", "--no-checkout", "--filter=blob:none", "--", repository, str(staging))
        git(staging, "fetch", "--depth", "1", "origin", source_ref)
        commit = git(staging, "rev-parse", "FETCH_HEAD^{commit}")
        # Keep submodule metadata paths short on Windows; check the full SHA below.
        destination = scratch_root / ("vlade_colab_" + commit[:16])
        if destination.exists():
            if (git(destination, "rev-parse", "HEAD") != commit
                    or git(destination, "remote", "get-url", "origin") != repository
                    or git(destination, "status", "--porcelain", "--untracked-files=normal")
                    or any(line[:1] in {"-", "+", "U"} for line in git(destination, "submodule", "status").splitlines()
                           if "third_party/ava_vla" in line)):
                raise ValueError("Existing source checkout changed; use a fresh source cache")
            return destination
        git(staging, "checkout", "--detach", commit)
        # The parent commit pins the AVA gitlink. Other baselines use their own environments.
        if (staging / ".gitmodules").is_file():
            git(staging, "submodule", "update", "--init", "--depth", "1", "--", "third_party/ava_vla")
        staging.rename(destination)
    return destination

CODE = stage_source(SOURCE_REPOSITORY, Path('/content/vlade_sources'), SOURCE_REF)
PROJECT_ROOT = CODE
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('실행 소스:', CODE, '\nGit commit:', SOURCE_COMMIT)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'), HF_TOKEN_PATH='/content/vlade_hf_token')
def run(args, env=None):
    import signal, threading, time
    deadline = float((env or ENV).get('VLADE_DEADLINE_UNIX', '0'))
    if deadline and deadline - time.time() <= 60:
        raise RuntimeError('8시간 예산이 끝났습니다. 새 작업을 시작하지 않습니다.')
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timers = []
    if deadline:
        for delay, callback in [(max(0, deadline-time.time()-60), lambda: process.send_signal(signal.SIGINT)),
                                (max(0, deadline-time.time()), process.kill)]:
            timer = threading.Timer(delay, callback)
            timer.daemon = True
            timer.start()
            timers.append(timer)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('실행에 실패했거나 시간 예산이 끝났습니다. 위 결과를 확인하십시오.')
    finally:
        for timer in timers:
            timer.cancel()
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

# Git supplies source. HF supplies every external model/data input.
ASSET_CACHE = Path('/content/vlade_hf_assets')
ASSET_WORK_ROOT = Path('/content/vlade_asset_work')
RUNS_ROOT = Path('/content/vlade_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
_HF_FETCHED = set()
ENV.update(VLADE_ASSET_CACHE=str(ASSET_CACHE),
           VLADE_ASSET_REGISTRY=str(CODE/'configs/storage/hf_assets.lock.json'),
           HF_HOME='/content/vlade_hf_home', HF_TOKEN_PATH='/content/vlade_hf_token')
os.environ.update({key: ENV[key] for key in ('VLADE_ASSET_CACHE','VLADE_ASSET_REGISTRY')})

def hf_input(key, python=None, *, fetch=True):
    from vlade.storage.assets import asset_record, cache_path
    record = asset_record(key, Path(ENV['VLADE_ASSET_REGISTRY']))
    target = cache_path(key, record, ASSET_CACHE)
    if fetch and (key, str(target)) not in _HF_FETCHED:
        if python is None:
            raise ValueError('Select the CPU/model Python environment before fetching HF assets')
        run([python, 'scripts/data/hf_assets.py', 'fetch', key,
             '--registry', ENV['VLADE_ASSET_REGISTRY'], '--cache', ASSET_CACHE], env=ENV)
        _HF_FETCHED.add((key, str(target)))
    return target

def hf_path(value, python=None, *, fetch=True):
    value = str(value)
    if not value.startswith('hfasset:'):
        raise ValueError('External inputs must use hfasset:// keys, not Drive/local paths')
    return hf_input(value[len('hfasset:'):].strip('/\\'), python, fetch=fetch)

In [ ]:
# HF 설정 후 기존 학습 셀을 실행하면 checkpoint 저장 시 자동 전송합니다.
# 저장소는 HF 웹에서 Private로 먼저 만들고 해당 저장소 쓰기 토큰을 Secrets에 등록하십시오.
if 'ENV' not in globals():
    ENV = os.environ.copy()
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
ENV['HF_TOKEN'] = os.environ['HF_TOKEN']
import datetime, uuid
ENABLE_HUB_UPLOAD = globals().get('ENABLE_HUB_UPLOAD', True)
HF_REPO_ID = 'VLADE-konkuk/VLADE-checkpoints'  # 조직 저장소; 이전 런타임 값도 갱신합니다.
HF_RUN_GROUP = globals().get('HF_RUN_GROUP', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])  # 재개 시 기존 group 지정
if ENABLE_HUB_UPLOAD:
    if not HF_RUN_GROUP:
        raise ValueError('HF_RUN_GROUP을 지정하십시오. pilot/main은 출력 폴더명으로 구분합니다.')
    os.environ['VLADE_HUB_REPO'] = HF_REPO_ID
    os.environ['VLADE_HUB_RUN_ID'] = HF_RUN_GROUP
    os.environ['HF_TOKEN_PATH'] = '/content/vlade_hf_token'
    # 모델 캐시는 기존 위치를 유지하고 토큰 파일은 런타임 로컬에만 둡니다.
    ENV.update(HF_TOKEN=os.environ['HF_TOKEN'], VLADE_HUB_REPO=HF_REPO_ID,
               VLADE_HUB_RUN_ID=HF_RUN_GROUP, HF_TOKEN_PATH='/content/vlade_hf_token')
    print('Private Hub 업로드 설정:', HF_REPO_ID, HF_RUN_GROUP)
else:
    for key in ('VLADE_HUB_REPO', 'VLADE_HUB_RUN_ID'):
        ENV.pop(key, None)
        os.environ.pop(key, None)
    print('Hub 업로드 꺼짐: HF_RUN_GROUP 설정 후 ENABLE_HUB_UPLOAD=True로 이 셀을 다시 실행하십시오.')

In [ ]:
import tomllib
run(['git', '-C', CODE, 'submodule', 'update', '--init', '--depth', '1', '--', 'third_party/openvla_oft'])
upstream = CODE / 'third_party/openvla_oft'
dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
dependencies = [dep for dep in dependencies if not dep.startswith(('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
dependencies += [
    'transformers @ git+https://github.com/moojink/transformers-openvla-oft.git@bc339d9ad707454c0c115970db43c260067c61ab',
    'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
    'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
    'huggingface_hub==0.36.2', 'accelerate==1.15.0']
OFT_PY = environment('vlade_oft_fft_py311', '3.11', dependencies)
run([UV, 'pip', 'install', '--python', OFT_PY, '--no-deps', '-e', upstream])
run([OFT_PY, '-c', "import torch,transformers,tensorflow_datasets; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,transformers.__version__,torch.cuda.get_device_name(0))"])
(ROOT / 'oft_environment_freeze.txt').write_text(subprocess.check_output(
    [str(UV), 'pip', 'freeze', '--python', str(OFT_PY)], text=True))

In [ ]:
PREPARED = hf_input('prepared_d_seed42', OFT_PY)
from vlade.data.episode_data import EpisodeData
data = EpisodeData(PREPARED)
assert data.manifest['counts'] == {'train': 11477, 'validation': 1428, 'test': 1442}
assert data.manifest['canonical_manifest_sha256'] == '487d2be1039ba6a4facc3ba44785ba5159245dfe53462a1d5a41ba8230446e80'
config = json.loads((CODE / 'configs/experiments/libero_plus_fft.json').read_text())
output = RUNS_ROOT / 'openvla_oft_fft/oft-one-step-seed42'
config['fft_defaults'].update(
    data_root=str(ASSET_WORK_ROOT / 'data/processed/libero_plus_tfds'), prepared_data_root=str(PREPARED),
    image_aug=False, batch_size=1, grad_accumulation_steps=1, max_steps=1, save_steps=1,
    save_optimizer_state=True, eval_steps=0)
config['fft_runs']['openvla_oft'].update(
    checkpoint=str(hf_input('openvla_oft_t0', OFT_PY)),
    output_dir=str(output))
configs = RUNS_ROOT / 'run-configs'; configs.mkdir(exist_ok=True)
config_path = configs / 'openvla-oft-one-step-seed42.json'
config_path.write_text(json.dumps(config, indent=2))
run([OFT_PY, 'scripts/train/run_fft.py', '--model', 'openvla_oft', '--config', config_path, '--preflight'])
if not (output / 'step-0000001/COMPLETE').is_file():
    run([OFT_PY, 'scripts/train/run_fft.py', '--model', 'openvla_oft', '--config', config_path])
run([OFT_PY, 'scripts/train/check_fft_checkpoint.py', '--checkpoint', output / 'step-0000001',
     '--data', PREPARED, '--config', config_path, '--report', output / 'reload_report.json'])